# 📊 APA vs Pure FP32 Baseline — Benchmark & Comparative Analyzer

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RedSafir/Adaptive-Precision-Ascension/blob/main/analyze_results.ipynb)

Notebook ini khusus dirancang untuk **membandingkan performa pelatihan (Benchmark Head-to-Head)** antara:
1. **APA (Adaptive Precision Ascension)**: Dimulai di FP8 dan eskalasi adaptif sesuai kebutuhan numerik.
2. **Pure FP32 Baseline**: Pelatihan konvensional murni menggunakan presisi 32-bit (kontrol standar).

### 🎯 Metrik Komparasi Utama:
- **Loss Convergence**: Perbandingan Train Loss dan Test/Val Loss antar-metode.
- **Accuracy Trajectory**: Perbandingan Train Accuracy dan Test/Val Accuracy (%) + Peak Accuracy.
- **Training Time & Throughput**: Waktu per epoch (detik), total waktu komputasi, dan rasio akselerasi (*Speedup Multiplier*).
- **Precision Drift**: Dinamika transisi presisi layer APA sepanjang epoch.
- **Executive Head-to-Head Summary Table**: Tabel komparasi lengkap siap diekspor ke paper/laporan.

## 1. Setup Environment & Ingesti Data (Google Drive / Local)
Sel ini menghubungkan Google Colab ke Google Drive (`/content/drive/MyDrive/result/`) dan memuat file log APA dan Baseline.

In [ ]:
import os
import json
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Styling visualisasi paper-ready
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 13
plt.rcParams['axes.titleweight'] = 'bold'
plt.rcParams['axes.labelweight'] = 'bold'

# Deteksi Google Colab & Auto-Mount Google Drive
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("💻 Terdeteksi di Google Colab. Menghubungkan ke Google Drive...")
    try:
        from google.colab import drive
        drive.mount('/content/drive')
        print("✅ Google Drive berhasil di-mount di /content/drive!")
    except Exception as e:
        print(f"ℹ️ Google Drive info: {e}")

# Pola pencarian file log
search_patterns = [
    '/content/drive/MyDrive/result/*.jsonl',
    '/content/drive/My Drive/result/*.jsonl',
    '/content/drive/MyDrive/*.jsonl',
    'result/*.jsonl',
    '*.jsonl'
]

all_detected_files = []
for pat in search_patterns:
    all_detected_files.extend(glob.glob(pat))

all_detected_files = list(dict.fromkeys(all_detected_files))
# Filter file forensik (fokus pada file epoch metrics)
metric_files = [f for f in all_detected_files if 'forensic' not in os.path.basename(f).lower()]

print(f"📁 Total file log training terdeteksi: {len(metric_files)}")
for idx, f in enumerate(metric_files):
    size_kb = os.path.getsize(f) / 1024 if os.path.exists(f) else 0
    print(f"  [{idx:2d}] {f:<45s} ({size_kb:6.1f} KB)")

# Deteksi otomatis file APA dan file Baseline FP32
apa_candidates = [f for f in metric_files if 'apa' in os.path.basename(f).lower()]
fp32_candidates = [f for f in metric_files if 'fp32' in os.path.basename(f).lower() or 'baseline' in os.path.basename(f).lower()]

APA_FILE = apa_candidates[0] if apa_candidates else (metric_files[0] if metric_files else None)
FP32_FILE = fp32_candidates[0] if fp32_candidates else (metric_files[1] if len(metric_files) > 1 else None)

print(f"\n👉 File Log APA Utama       : {APA_FILE}")
print(f"👉 File Log FP32 Baseline   : {FP32_FILE}")

## 2. Ingesti Data Log & Rekonstruksi Metrik
Mem-parsing riwayat pelatihan per epoch (Loss, Akurasi, Waktu, dan Sebaran Presisi).

In [ ]:
def parse_training_log(filepath):
    if not filepath or not os.path.exists(filepath):
        return pd.DataFrame(), []
    
    epochs = []
    escalations = []
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                d = json.loads(line)
            except json.JSONDecodeError:
                continue
                
            # Record epoch summary
            if 'epoch' in d and ('train_loss' in d or 'test_loss' in d):
                rec = dict(d)
                # Normalisasi persentase akurasi
                if 'train_acc' in rec:
                    rec['train_acc_pct'] = rec['train_acc'] * 100 if rec['train_acc'] <= 1.0 else rec['train_acc']
                if 'test_acc' in rec:
                    rec['test_acc_pct'] = rec['test_acc'] * 100 if rec['test_acc'] <= 1.0 else rec['test_acc']
                epochs.append(rec)
            elif d.get('event') == 'escalation':
                escalations.append(d)
                
    df_epochs = pd.DataFrame(epochs).drop_duplicates(subset=['epoch']).sort_values('epoch') if epochs else pd.DataFrame()
    return df_epochs, escalations

df_apa, esc_apa = parse_training_log(APA_FILE)
df_fp32, _ = parse_training_log(FP32_FILE)

print(f"✅ Berhasil memuat:")
print(f"  • Log APA          : {len(df_apa)} epoch tercatat (Total eskalasi: {len(esc_apa)})")
print(f"  • Log FP32 Baseline: {len(df_fp32)} epoch tercatat")

## 3. Komparasi Kurva Konvergensi Loss (Train & Test Loss)
Membandingkan apakah metode **APA** konvergen secepat dan semulus **Pure FP32 Baseline**.

In [ ]:
if not df_apa.empty or not df_fp32.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Panel Kiri: Training Loss
    if not df_apa.empty and 'train_loss' in df_apa.columns:
        ax1.plot(df_apa['epoch'], df_apa['train_loss'], label='APA (Adaptive Precision)', color='#2563EB', linewidth=2.5, marker='o')
    if not df_fp32.empty and 'train_loss' in df_fp32.columns:
        ax1.plot(df_fp32['epoch'], df_fp32['train_loss'], label='Pure FP32 Baseline', color='#64748B', linewidth=2.5, linestyle='--', marker='s')
    
    ax1.set_title('Training Loss Convergence', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Loss', fontweight='bold')
    ax1.legend(loc='upper right', frameon=True)
    ax1.grid(True, alpha=0.3)
    
    # Panel Kanan: Test/Validation Loss
    if not df_apa.empty and 'test_loss' in df_apa.columns:
        ax2.plot(df_apa['epoch'], df_apa['test_loss'], label='APA (Adaptive Precision)', color='#2563EB', linewidth=2.5, marker='o')
    if not df_fp32.empty and 'test_loss' in df_fp32.columns:
        ax2.plot(df_fp32['epoch'], df_fp32['test_loss'], label='Pure FP32 Baseline', color='#64748B', linewidth=2.5, linestyle='--', marker='s')
        
    ax2.set_title('Test / Validation Loss Convergence', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch', fontweight='bold')
    ax2.set_ylabel('Loss', fontweight='bold')
    ax2.legend(loc='upper right', frameon=True)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Data loss belum tersedia.")

## 4. Komparasi Kurva Akurasi (Train & Test Accuracy)
Mengevaluasi apakah **APA** mampu mempertahankan akurasi generalisasi yang setara dengan FP32.

In [ ]:
if not df_apa.empty or not df_fp32.empty:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # Panel Kiri: Training Accuracy
    if not df_apa.empty and 'train_acc_pct' in df_apa.columns:
        ax1.plot(df_apa['epoch'], df_apa['train_acc_pct'], label='APA (Adaptive Precision)', color='#2563EB', linewidth=2.5, marker='o')
    if not df_fp32.empty and 'train_acc_pct' in df_fp32.columns:
        ax1.plot(df_fp32['epoch'], df_fp32['train_acc_pct'], label='Pure FP32 Baseline', color='#64748B', linewidth=2.5, linestyle='--', marker='s')
        
    ax1.set_title('Training Accuracy Trajectory', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Accuracy (%)', fontweight='bold')
    ax1.legend(loc='lower right', frameon=True)
    ax1.grid(True, alpha=0.3)
    
    # Panel Kanan: Test Accuracy & Peak Annotation
    if not df_apa.empty and 'test_acc_pct' in df_apa.columns:
        ax2.plot(df_apa['epoch'], df_apa['test_acc_pct'], label='APA (Adaptive Precision)', color='#10B981', linewidth=2.5, marker='o')
        best_apa = df_apa['test_acc_pct'].max()
        best_apa_ep = df_apa.loc[df_apa['test_acc_pct'].idxmax(), 'epoch']
        ax2.scatter([best_apa_ep], [best_apa], color='#10B981', s=120, zorder=6)
        ax2.annotate(f'Peak APA: {best_apa:.2f}% (Ep {best_apa_ep})', xy=(best_apa_ep, best_apa),
                     xytext=(best_apa_ep, best_apa + 1.5), fontweight='bold', color='#047857',
                     arrowprops=dict(arrowstyle='->', color='#047857', lw=1.5))
                     
    if not df_fp32.empty and 'test_acc_pct' in df_fp32.columns:
        ax2.plot(df_fp32['epoch'], df_fp32['test_acc_pct'], label='Pure FP32 Baseline', color='#64748B', linewidth=2.5, linestyle='--', marker='s')
        best_fp32 = df_fp32['test_acc_pct'].max()
        best_fp32_ep = df_fp32.loc[df_fp32['test_acc_pct'].idxmax(), 'epoch']
        ax2.scatter([best_fp32_ep], [best_fp32], color='#64748B', s=120, zorder=6)
        ax2.annotate(f'Peak FP32: {best_fp32:.2f}% (Ep {best_fp32_ep})', xy=(best_fp32_ep, best_fp32),
                     xytext=(best_fp32_ep, best_fp32 - 3.5), fontweight='bold', color='#334155',
                     arrowprops=dict(arrowstyle='->', color='#334155', lw=1.5))
                     
    ax2.set_title('Test / Validation Accuracy Comparison', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch', fontweight='bold')
    ax2.set_ylabel('Accuracy (%)', fontweight='bold')
    ax2.legend(loc='lower right', frameon=True)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Data akurasi belum tersedia.")

## 5. Komparasi Waktu Training & Throughput Speedup
Membandingkan efisiensi waktu eksekusi per-epoch dan total durasi komputasi.

In [ ]:
if not df_apa.empty and not df_fp32.empty and 'epoch_time_sec' in df_apa.columns and 'epoch_time_sec' in df_fp32.columns:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    # 1. Waktu per Epoch (Detik)
    ax1.plot(df_apa['epoch'], df_apa['epoch_time_sec'], label='APA Epoch Duration', color='#2563EB', linewidth=2.5, marker='o')
    ax1.plot(df_fp32['epoch'], df_fp32['epoch_time_sec'], label='FP32 Baseline Epoch Duration', color='#DC2626', linewidth=2.5, linestyle='--', marker='s')
    ax1.set_title('Waktu Komputasi per Epoch', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Epoch', fontweight='bold')
    ax1.set_ylabel('Waktu (Detik)', fontweight='bold')
    ax1.legend(loc='upper right', frameon=True)
    ax1.grid(True, alpha=0.3)
    
    # 2. Waktu Kumulatif Training (Menit)
    apa_cum_time = df_apa['epoch_time_sec'].cumsum() / 60
    fp32_cum_time = df_fp32['epoch_time_sec'].cumsum() / 60
    
    ax2.plot(df_apa['epoch'], apa_cum_time, label='APA Total Wall-Time', color='#2563EB', linewidth=2.5, marker='o')
    ax2.plot(df_fp32['epoch'], fp32_cum_time, label='FP32 Total Wall-Time', color='#DC2626', linewidth=2.5, linestyle='--', marker='s')
    
    # Area penghematan waktu
    min_len = min(len(apa_cum_time), len(fp32_cum_time))
    ax2.fill_between(df_apa['epoch'][:min_len], apa_cum_time[:min_len], fp32_cum_time[:min_len],
                     color='#10B981', alpha=0.25, label='Waktu yang Dihemat (Time Saved)')
                     
    ax2.set_title('Kumulatif Waktu Pelatihan', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Epoch', fontweight='bold')
    ax2.set_ylabel('Total Waktu (Menit)', fontweight='bold')
    ax2.legend(loc='upper left', frameon=True)
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
else:
    print("ℹ️ Data perbandingan waktu per-epoch lengkap belum tersedia pada kedua file.")

## 6. Evolusi Presisi Jaringan (Khusus Metode APA)
Memvisualisasikan dinamika *precision drift* dari awal FP8 hingga eskalasi ke FP16/TF32.

In [ ]:
if not df_apa.empty and 'precision_distribution' in df_apa.columns:
    drift_list = []
    for _, r in df_apa.iterrows():
        p = r['precision_distribution']
        if isinstance(p, dict):
            tot = p.get('fp8', 0) + p.get('fp16', 0) + p.get('tf32', 0)
            if tot > 0:
                drift_list.append({
                    'epoch': r['epoch'],
                    'FP8': (p.get('fp8', 0) / tot) * 100,
                    'FP16': (p.get('fp16', 0) / tot) * 100,
                    'TF32': (p.get('tf32', 0) / tot) * 100
                })
                
    if drift_list:
        df_drift = pd.DataFrame(drift_list)
        plt.figure(figsize=(12, 4.5))
        plt.stackplot(df_drift['epoch'], df_drift['FP8'], df_drift['FP16'], df_drift['TF32'],
                      labels=['FP8 (Maximum Throughput)', 'FP16 (Medium Dynamic Range)', 'TF32 (Full Precision Extension)'],
                      colors=['#10B981', '#3B82F6', '#EF4444'], alpha=0.85)
        plt.xlabel('Epoch', fontweight='bold')
        plt.ylabel('Komposisi Presisi Jaringan (%)', fontweight='bold')
        plt.ylim(0, 100)
        plt.title('APA Precision Drift Evolution: Transisi Presisi Sepanjang Pelatihan', fontsize=12, fontweight='bold')
        plt.legend(loc='upper right', frameon=True)
        plt.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.show()
else:
    print("ℹ️ Data sebaran presisi belum tersedia.")

## 7. 🏆 Tabel Ringkasan Eksekutif (Head-to-Head APA vs FP32 Baseline)
Tabel ringkasan metrik akhir, akurasi puncak, efisiensi waktu, dan rasio speedup.

In [ ]:
# Bangun tabel perbandingan komparatif
summary_rows = []

has_apa = not df_apa.empty
has_fp32 = not df_fp32.empty

# 1. Akurasi Akhir
apa_train_acc = f"{df_apa.iloc[-1]['train_acc_pct']:.2f}%" if has_apa and 'train_acc_pct' in df_apa.columns else '-'
fp32_train_acc = f"{df_fp32.iloc[-1]['train_acc_pct']:.2f}%" if has_fp32 and 'train_acc_pct' in df_fp32.columns else '-'
summary_rows.append({'Metrik': 'Final Train Accuracy', 'APA': apa_train_acc, 'Pure FP32 Baseline': fp32_train_acc})

apa_test_acc = f"{df_apa.iloc[-1]['test_acc_pct']:.2f}%" if has_apa and 'test_acc_pct' in df_apa.columns else '-'
fp32_test_acc = f"{df_fp32.iloc[-1]['test_acc_pct']:.2f}%" if has_fp32 and 'test_acc_pct' in df_fp32.columns else '-'
summary_rows.append({'Metrik': 'Final Test Accuracy', 'APA': apa_test_acc, 'Pure FP32 Baseline': fp32_test_acc})

# 2. Akurasi Puncak
apa_peak = f"{df_apa['test_acc_pct'].max():.2f}% (Ep {df_apa.loc[df_apa['test_acc_pct'].idxmax(), 'epoch']})" if has_apa and 'test_acc_pct' in df_apa.columns else '-'
fp32_peak = f"{df_fp32['test_acc_pct'].max():.2f}% (Ep {df_fp32.loc[df_fp32['test_acc_pct'].idxmax(), 'epoch']})" if has_fp32 and 'test_acc_pct' in df_fp32.columns else '-'
summary_rows.append({'Metrik': '⭐ Peak Test Accuracy', 'APA': apa_peak, 'Pure FP32 Baseline': fp32_peak})

# 3. Loss Akhir
apa_train_loss = f"{df_apa.iloc[-1]['train_loss']:.4f}" if has_apa and 'train_loss' in df_apa.columns else '-'
fp32_train_loss = f"{df_fp32.iloc[-1]['train_loss']:.4f}" if has_fp32 and 'train_loss' in df_fp32.columns else '-'
summary_rows.append({'Metrik': 'Final Train Loss', 'APA': apa_train_loss, 'Pure FP32 Baseline': fp32_train_loss})

apa_test_loss = f"{df_apa.iloc[-1]['test_loss']:.4f}" if has_apa and 'test_loss' in df_apa.columns else '-'
fp32_test_loss = f"{df_fp32.iloc[-1]['test_loss']:.4f}" if has_fp32 and 'test_loss' in df_fp32.columns else '-'
summary_rows.append({'Metrik': 'Final Test Loss', 'APA': apa_test_loss, 'Pure FP32 Baseline': fp32_test_loss})

# 4. Waktu Training
apa_avg_time = f"{df_apa['epoch_time_sec'].mean():.2f}s" if has_apa and 'epoch_time_sec' in df_apa.columns else '-'
fp32_avg_time = f"{df_fp32['epoch_time_sec'].mean():.2f}s" if has_fp32 and 'epoch_time_sec' in df_fp32.columns else '-'
summary_rows.append({'Metrik': 'Avg Time / Epoch', 'APA': apa_avg_time, 'Pure FP32 Baseline': fp32_avg_time})

apa_tot_time = f"{df_apa['epoch_time_sec'].sum()/60:.2f} min" if has_apa and 'epoch_time_sec' in df_apa.columns else '-'
fp32_tot_time = f"{df_fp32['epoch_time_sec'].sum()/60:.2f} min" if has_fp32 and 'epoch_time_sec' in df_fp32.columns else '-'
summary_rows.append({'Metrik': 'Total Training Time', 'APA': apa_tot_time, 'Pure FP32 Baseline': fp32_tot_time})

# 5. Speedup Multiplier
if has_apa and has_fp32 and 'epoch_time_sec' in df_apa.columns and 'epoch_time_sec' in df_fp32.columns:
    t_apa = df_apa['epoch_time_sec'].sum()
    t_fp32 = df_fp32['epoch_time_sec'].sum()
    speedup = t_fp32 / t_apa if t_apa > 0 else 1.0
    speedup_str = f"{speedup:.2f}x Faster" if speedup >= 1.0 else f"{1/speedup:.2f}x Slower"
else:
    speedup_str = '-'
summary_rows.append({'Metrik': '⚡ Speedup Multiplier', 'APA': speedup_str, 'Pure FP32 Baseline': '1.00x (Ref)'})

df_summary = pd.DataFrame(summary_rows)
print("=" * 75)
print("🏆 BENCHMARK COMPARISON: APA vs PURE FP32 BASELINE")
print("=" * 75)
display(df_summary)

# Ekspor ke CSV
output_csv = 'apa_vs_fp32_benchmark_summary.csv'
df_summary.to_csv(output_csv, index=False)
print(f"\n💾 Hasil tabel ringkasan berhasil disimpan ke: {output_csv}")
if IN_COLAB:
    from google.colab import files
    files.download(output_csv)